# Metabis case study: Compass Th17 workflow

This notebook applies the public Metabis workflow end to end: explicit Compass activity decoding, standalone sample metadata, metadata-derived Sample Groups, a pathogenic-Th17 Target versus non-pathogenic-Th17 Baseline comparison, and artifact-first plots.


## Before you run it

Install the repository with its development dependencies and generate the three Th17 example files as described in `demo/README.md`.


In [ ]:
# ruff: noqa: E402, I001
import os
import sys
import tempfile
import warnings
from pathlib import Path

import pandas as pd

from bokeh.io import output_notebook, show

os.environ.setdefault(
    "MPLCONFIGDIR",
    str(Path(tempfile.gettempdir()) / "metabis-matplotlib"),
)
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)
warnings.filterwarnings("ignore", message="IProgress not found.*")


def find_metabis_repo_root_for_import(start: Path) -> Path:
    direct_candidates = [start, *start.parents]
    nested_candidates = [candidate / "metabis" for candidate in direct_candidates]
    for candidate in [*direct_candidates, *nested_candidates]:
        if (candidate / "src" / "metabis").exists():
            return candidate
    raise FileNotFoundError("Could not locate the Metabis repository root.")


repo_root_for_import = find_metabis_repo_root_for_import(Path.cwd().resolve())
src_path = repo_root_for_import / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from metabis import (
    AnalysisWorkspace,
    ExperimentAnalysis,
    ExpressionData,
    ExpressionPreprocessingConfig,
    GroupComparison,
    GroupComparisonSpec,
    MetabolicModel,
    ReactionData,
    ReactionPreprocessingConfig,
    SampleMetadata,
    build_reduction_presentation,
    plot_reduction_artifact,
    plot_statistical_histograms,
    plot_subsystem_comparison,
)
from metabis.analysis_workspace import RevisionKind
from metabis.version import __version__ as metabis_version
from metabis.viz.metabolic_visualization import (
    EnrichmentConfig,
    GraphGenerationConfig,
    MetabolicGraphPipeline,
    PipelineConfig,
    RenderingConfig,
    render_graph_artifact,
)

output_notebook()


In [ ]:
repo_root = find_metabis_repo_root_for_import(Path.cwd().resolve())
data_dir = repo_root / "data"
required_example_files = (
    "Th17_Compass_reactions.tsv",
    "Th17_Compass_expression.tsv",
    "Th17_Compass_metadata.tsv",
)
missing = [name for name in required_example_files if not (data_dir / name).is_file()]
if missing:
    raise FileNotFoundError(
        "Missing generated example files: "
        + ", ".join(missing)
        + ". Run `python data/download_and_process_compass.py`."
    )

reaction_data_path = data_dir / "Th17_Compass_reactions.tsv"
expression_data_path = data_dir / "Th17_Compass_expression.tsv"
metadata_path = data_dir / "Th17_Compass_metadata.tsv"
target_group_id = "Th17p"
baseline_group_id = "Th17n"
analysis_settings = {
    "pval_threshold": 0.10,
    "cohens_d_threshold": 0.0,
    "log2_fc_threshold": 0.0,
    "mean_difference_threshold": 0.0,
    "keep_reversed": True,
}

print(f"Metabis version: {metabis_version}")
print(f"Target group: {target_group_id}")
print(f"Baseline group: {baseline_group_id}")


In [ ]:
model = MetabolicModel(use_cache=True)

alignment_file_path = model.get_alignment_file_path("RECON2")
model.apply_alignment_corrections(alignment_file_path)

print(
    f"Model loaded: {len(model.get_reactions())} reactions and {len(model.get_genes())} genes"
)


## Load the Th17 data

This case study uses the same loading configuration as the staged demos and moves directly into the Th17 application.


In [ ]:
reaction_loading_options = {
    "data_mode": "dual",
    "reaction_name_format": "rxnRecon3DID",
    "csv_separator": "\t",
    "samples_in": "columns",
    "check_old_identifiers": True,
    "preprocessing": ReactionPreprocessingConfig(compass_activity=True),
}
expression_loading_options = {
    "gene_format": "geneSymbols",
    "preprocessing": ExpressionPreprocessingConfig(normalize_total=True),
    "data_format": "csv",
    "csv_separator": "\t",
    "samples_in": "columns",
}


In [ ]:
reaction_data, _ = ReactionData.create_with_validation(
    str(reaction_data_path),
    metabolic_model=model,
    **reaction_loading_options,
)

reaction_df = reaction_data.get_reaction_df()

print(
    f"Reaction matrix: {reaction_df.shape[0]} samples x {reaction_df.shape[1]} reaction features"
)


In [ ]:
expression_data, _ = ExpressionData.create_with_validation(
    str(expression_data_path),
    gene_list=model.get_genes(),
    metabolic_model=model,
    **expression_loading_options,
)

expression_df = expression_data.get_expression_df()

print(
    f"Expression matrix: {expression_df.shape[0]} samples x {expression_df.shape[1]} genes"
)


In [ ]:
metadata = SampleMetadata.from_table(
    metadata_path,
    sample_id_column="sample",
)
dm = AnalysisWorkspace(
    metabolic_model=model,
    reaction_data=reaction_data,
    expression_data=expression_data,
)
dm.update_sample_metadata(metadata)
dm.load_groups_from_metadata(["cell_type"])

group_ids = sorted(dm.get_group_ids())
sample_counts = {
    group_id: len(dm.get_group_samples(group_id)) for group_id in group_ids
}
assert target_group_id in group_ids and baseline_group_id in group_ids
print(f"Metadata-derived groups: {group_ids}")
print(f"Samples per group: {sample_counts}")


## Group structure before comparison

A quick reaction-space PCA is a useful bridge between grouped data setup and the differential comparison.


In [ ]:
analysis_dependencies = frozenset(
    {
        RevisionKind.DATASET,
        RevisionKind.METADATA,
        RevisionKind.GROUPS,
        RevisionKind.PREPROCESSING,
    }
)
analysis_snapshot = dm.capture_analysis_snapshot(analysis_dependencies)

analysis = ExperimentAnalysis(dm, auto_run=False)
analysis_outcome = analysis.run_analysis(
    include_pca=True,
    pca_data_types=["reactions"],
    n_components=4,
    n_variable_features=5000,
)
assert analysis_outcome["status"] == "success", analysis_outcome["messages"]
pca_result = analysis.get_reduction_result("PCA", "reactions")
assert pca_result is not None
pca_artifact = pca_result.build_artifact(
    provenance=analysis_snapshot.provenance,
    source_fingerprints=dict(dm.source_fingerprints),
)
analysis.install_reduction_artifact(pca_artifact)
dm.install_experiment_analysis(analysis_snapshot.provenance, analysis)
dm.refresh_experiment_analysis_presentation()
dm.set_analysis_group(target_group_id)

print(f"Prepared experiment analysis for Target: {target_group_id}")


In [ ]:
pca_presentation = build_reduction_presentation(
    dm,
    pca_artifact,
    width=950,
    height=700,
)
show(plot_reduction_artifact(pca_artifact, pca_presentation))


## Group comparison: pathogenic Th17 Target vs non-pathogenic Th17 Baseline

The comparison specification and resolved membership make the orientation explicit before any statistical test runs.


In [ ]:
comparison_spec = GroupComparisonSpec.pairwise(
    (target_group_id,),
    (baseline_group_id,),
    target_label="Pathogenic Th17",
    baseline_label="Non-pathogenic Th17",
)
group_comparison = GroupComparison(
    spec=comparison_spec,
    test="wilcoxon",
    correction="fdr_bh",
)
group_comparison.run_test(dm, for_genes=False)
group_comparison.run_test(dm, for_genes=True)


In [ ]:
significant_reactions, _ = group_comparison.define_significant_elements(
    pval_threshold=analysis_settings["pval_threshold"],
    cohens_d_threshold=analysis_settings["cohens_d_threshold"],
    log2_fc_threshold=analysis_settings["log2_fc_threshold"],
    mean_difference_threshold=analysis_settings["mean_difference_threshold"],
    keep_reversed=analysis_settings["keep_reversed"],
    for_genes=False,
)
significant_genes, _ = group_comparison.define_significant_elements(
    pval_threshold=analysis_settings["pval_threshold"],
    cohens_d_threshold=analysis_settings["cohens_d_threshold"],
    log2_fc_threshold=analysis_settings["log2_fc_threshold"],
    mean_difference_threshold=analysis_settings["mean_difference_threshold"],
    keep_reversed=analysis_settings["keep_reversed"],
    for_genes=True,
)


In [ ]:
comparison_snapshot = dm.capture_analysis_snapshot(analysis_dependencies)
comparison_artifact = group_comparison.build_artifact(
    provenance=comparison_snapshot.provenance,
    source_fingerprints=dict(dm.source_fingerprints),
    keep_reversed=analysis_settings["keep_reversed"],
)
dm.install_group_comparison_artifact(
    comparison_snapshot.provenance,
    comparison_artifact,
)
dm.refresh_group_comparison_presentation()

reaction_results = comparison_artifact.get_test_results(for_genes=False)
gene_results = comparison_artifact.get_test_results(for_genes=True)
print(
    f"Comparison: {comparison_artifact.membership.target_label} minus "
    f"{comparison_artifact.membership.baseline_label}"
)
print(f"Reaction-level results: {reaction_results.shape[0]} rows")
print(f"Gene-level results: {gene_results.shape[0]} rows")
print(f"Significant reactions: {len(significant_reactions)}")
print(f"Significant genes: {len(significant_genes)}")


## Global diagnostics

Before moving to pathway interpretation, inspect the overall distribution of the comparison statistics and the subsystem ranking derived from the current comparison.


In [ ]:
plot_statistical_histograms(
    dm,
    parameters_to_plot=["pval", "log2_fc", "cohens_d"],
    return_plot=False,
    show_thresholds=True,
    for_genes=False,
)

In [ ]:
subsystem_stats = dm.subsystem_analysis.rank_subsystems_by_delta_score(
    for_genes=False
)

plot_subsystem_comparison(
    dm,
    subsystem_stats=subsystem_stats,
    metric="cohens_d",
    width=900,
    return_plot=False,
)


def show_graph_artifact(
    element_name,
    graph_type,
    *,
    layout_engine,
    height,
):
    graph_snapshot = dm.capture_analysis_snapshot(analysis_dependencies)
    pipeline = MetabolicGraphPipeline(
        model,
        PipelineConfig(
            generation_config=GraphGenerationConfig(
                use_human1_map=True,
                layout_engine=layout_engine,
            ),
            enrichment_config=EnrichmentConfig(color_metric="cohens_d"),
            show_plot=False,
            enable_timing=True,
        ),
    )
    artifact = pipeline.build_artifact(
        element_name,
        graph_type,
        data_manager=dm,
        provenance=graph_snapshot.provenance,
        source_fingerprints=dict(dm.source_fingerprints),
    )
    rendered = render_graph_artifact(
        artifact,
        RenderingConfig(
            height=height,
            sizing_mode="scale_both",
            aspect_ratio=2,
            zoom=1,
        ),
    )
    show(rendered.plot)
    print(
        f"Rendered {artifact.display_title}: "
        f"{artifact.node_count} nodes, {artifact.edge_count} edges"
    )
    return artifact


## Global metabolic network

Positive red values mean higher activity in the pathogenic-Th17 Target; negative blue values mean higher activity in the non-pathogenic-Th17 Baseline.


The overview provides a pathway-level map of the exact installed comparison artifact. Interpret individual regions together with their reaction-level statistics and provenance.


In [ ]:
overview_artifact = show_graph_artifact(
    None,
    "metabolic_network",
    layout_engine="fdp",
    height=850,
)


## Pathway view: Glycolysis / Gluconeogenesis

This subsystem should show the expected Th17p-enriched glycolytic signal.


In [ ]:
glycolysis_artifact = show_graph_artifact(
    "Glycolysis / Gluconeogenesis",
    "subsystem",
    layout_engine="dot",
    height=750,
)


## Pathway view: the TCA cycle

The key pattern here is not uniform activation, but a split or "broken" TCA configuration across individual reactions.


In [ ]:
tca_artifact = show_graph_artifact(
    "Tricarboxylic acid cycle and glyoxylate/dicarboxylate metabolism",
    "subsystem",
    layout_engine="dot",
    height=850,
)


The TCA view should display red segments separated by more neutral reactions, reproducing the graph-level signature described in the case study.

## Integrated multi-omics around putrescine

The final subsystem couples reaction-level effects with gene context in arginine and proline metabolism, with particular attention to the route from arginine toward putrescine.


In [ ]:
gene_metrics = [
    "adjusted_pval",
    "log2_fc",
    "cohens_d",
    "mean_difference",
]
gene_summary_rows = []

for gene_symbol in ["SAT1", "ODC1"]:
    ensembl_id = model.translate_gene(
        gene_symbol,
        input_format="geneSymbols",
        output_format="ensmbl_id",
    )
    if ensembl_id not in gene_results.index:
        raise KeyError(
            f"{gene_symbol} ({ensembl_id}) was not found in the gene-level test results"
        )

    gene_row = gene_results.loc[ensembl_id, gene_metrics].copy()
    gene_row["ensmbl_id"] = ensembl_id
    gene_row["is_significant"] = ensembl_id in comparison_artifact.significant_genes
    gene_row.name = gene_symbol
    gene_summary_rows.append(gene_row)

gene_summary = pd.DataFrame(gene_summary_rows)[
    [
        "ensmbl_id",
        "adjusted_pval",
        "log2_fc",
        "cohens_d",
        "mean_difference",
        "is_significant",
    ]
]
gene_summary


## Case-study findings

The workflow keeps every interpretive surface tied to explicit inputs: generated Compass activity scores, metadata-derived Sample Groups, exact Target/Baseline membership, immutable reduction and comparison artifacts, and Graph Artifacts rendered through a separate Bokeh adapter.

Use the exported full-precision tables beside the pathway views when drawing biological conclusions.
